# **Assumptions (A1)-(A5) of van den Driessche and Watmough with `check_assumptions()`**

The next-generation matrix method writes each compartment as
$$\dot x_i = \mathcal{F}_i(x) - \mathcal{V}_i(x), \qquad \mathcal{V}_i = \mathcal{V}_i^- - \mathcal{V}_i^+,$$
where $\mathcal{F}_i$ are the new infections, $\mathcal{V}_i^+$ the other inflows and $\mathcal{V}_i^-$ the outflows. Theorem 2 of van den Driessche and Watmough (2002) guarantees that **$\mathcal{R}_0 = \rho(FV^{-1})$ is a threshold** (the DFE is locally asymptotically stable if $\mathcal{R}_0 < 1$ and unstable if $\mathcal{R}_0 > 1$) only if five assumptions hold:

| | Statement |
|:--:|---|
| (A1) | If $x \ge 0$, then $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ |
| (A2) | If $x_i = 0$, then $\mathcal{V}_i^- = 0$ |
| (A3) | $\mathcal{F}_i = 0$ for the uninfected compartments ($i > m$) |
| (A4) | If $x \in X_s$ (all infected compartments at zero), then $\mathcal{F}_i = 0$ and $\mathcal{V}_i^+ = 0$ for $i \le m$ |
| (A5) | With $\mathcal{F} = 0$, the eigenvalues of $Df(x_0)$ have negative real part: those of $V$ and $J_4$ have positive real part |

`model.check_assumptions()` decides each assumption as **holds**, **fails** or **undecided**, and states the basis:

* **by construction**: guaranteed by the way pyR0compute builds the model, as (A3);
* **symbolic proof**: valid for every positive value of the parameters;
* **at the given values**: decided at the point `values={...}`;
* **random samples**: if the violation appears in every parameter sample, *fails*; if the assumption holds in every sample but could not be proved, or holds only in some, *undecided* (with `values=` it is decided at a point).

The report is returned in LaTeX, ready for a paper (`to_latex()`), or in Markdown for the notebook.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/05_vdw_assumptions.ipynb)

In [1]:
%pip install pyR0compute                      # Install pyR0compute from PyPI

In [2]:
import warnings                               # Some models have several DFEs (warning)
import numpy as np                            # Numerical computation
import sympy as sp                            # Symbolic computation
from pyr0compute import R0Model               # Main class of the library
from IPython.display import Markdown, display # Show the report


def section(report, key):
    '''Show only the section of one assumption (for example "A4") of the Markdown report.'''
    text = report.to_latex("markdown")
    start = text.find(f"### ({key})")
    end = text.find("\n### ", start + 1)
    display(Markdown(text[start:end]))

## 1. A model that satisfies the five assumptions

The T-helper lymphocyte model of Cuesta-Herrera et al. (2025), *Math. Biosci. Eng.* 22(11):2807-2825. The five assumptions are proved for every positive value of the parameters. In (A5), $J_4$ has eigenvalues $d$ and $c$, the same that appear in the proof of Theorem 1 of the paper.

When the report is evaluated in a cell, Jupyter shows it in full: summary table, decomposition $\mathcal{F}_i$, $\mathcal{V}_i^+$, $\mathcal{V}_i^-$, the details of each assumption, the consequences of Lemma 1 ($F \ge 0$ and $V$ a nonsingular M-matrix) and the conclusion.

In [3]:
model_th = R0Model('''
    dE/dt  = lambda - d*E - kappa*E*V
    dI/dt  = kappa*E*V - alpha*I - beta*Th*I
    dV/dt  = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
''', infected=["I", "V"])

report_th = model_th.check_assumptions()
report_th

## Assumptions of van den Driessche and Watmough (2002)

Infected compartments ($i \le m$): $I, V$. No parameter values given: each assumption is proved for every positive parameter value when possible, and otherwise explored with random samples.

| Assumption | Statement | Result | Basis |
|:--:|---|:--:|---|
| (A1) | non-negative flows | **holds** | symbolic proof for all positive parameters |
| (A2) | no outflow from an empty compartment | **holds** | symbolic proof for all positive parameters |
| (A3) | no new infections in uninfected compartments | **holds** | by construction |
| (A4) | the disease-free subspace is invariant | **holds** | symbolic proof for all positive parameters |
| (A5) | the DFE is stable without new infections | **holds** | symbolic proof for all positive parameters |

### Decomposition used

$\dot x_i = \mathcal{F}_i - \mathcal{V}_i$, with $\mathcal{V}_i = \mathcal{V}_i^- - \mathcal{V}_i^+$ ($\mathcal{V}_i^+$: other inflows; $\mathcal{V}_i^-$: outflows).

$$
\begin{aligned}
\mathcal{F}_{E} &= 0, & \mathcal{V}_{E}^+ &= \lambda, & \mathcal{V}_{E}^- &= E V \kappa + E d \\
\mathcal{F}_{I} &= E V \kappa, & \mathcal{V}_{I}^+ &= 0, & \mathcal{V}_{I}^- &= I Th \beta + I \alpha \\
\mathcal{F}_{V} &= 0, & \mathcal{V}_{V}^+ &= I \nu, & \mathcal{V}_{V}^- &= Th V \tau + V \mu \\
\mathcal{F}_{Th} &= 0, & \mathcal{V}_{Th}^+ &= I Th \gamma + b, & \mathcal{V}_{Th}^- &= Th c
\end{aligned}
$$

### (A1) non-negative flows

If $x \ge 0$, then $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ for $i = 1,\dots,n$.

Result: **holds** (symbolic proof for all positive parameters).

* $\mathcal{V}_i^+$ and $\mathcal{V}_i^-$ collect the negative and positive terms of $\mathcal{V}_i$; they are non-negative when every term has a definite sign.

### (A2) no outflow from an empty compartment

If $x_i = 0$, then $\mathcal{V}_i^- = 0$.

Result: **holds** (symbolic proof for all positive parameters).

* $\left.\mathcal{V}_{E}^-\right\rvert_{E = 0} = 0$: **holds**
* $\left.\mathcal{V}_{I}^-\right\rvert_{I = 0} = 0$: **holds**
* $\left.\mathcal{V}_{V}^-\right\rvert_{V = 0} = 0$: **holds**
* $\left.\mathcal{V}_{Th}^-\right\rvert_{Th = 0} = 0$: **holds**

### (A3) no new infections in uninfected compartments

$\mathcal{F}_i = 0$ for $i > m$ (uninfected compartments).

Result: **holds** (by construction).

* pyR0compute defines $\mathcal{F}_i$ only for the infected compartments and rejects new-infection terms for uninfected ones.

### (A4) the disease-free subspace is invariant

If $x \in X_s$, then $\mathcal{F}_i(x) = 0$ and $\mathcal{V}_i^+(x) = 0$ for $i = 1,\dots,m$.

Result: **holds** (symbolic proof for all positive parameters).

* $\left.\mathcal{F}_{I}\right\rvert_{X_s} = 0$: **holds**
* $\left.\mathcal{V}_{I}^+\right\rvert_{X_s} = 0$: **holds**
* $\left.\mathcal{F}_{V}\right\rvert_{X_s} = 0$: **holds**
* $\left.\mathcal{V}_{V}^+\right\rvert_{X_s} = 0$: **holds**
* $X_s = \{x \ge 0 : x_i = 0,\ i \le m\}$; the uninfected compartments are free, not fixed at $x_0$.

### (A5) the DFE is stable without new infections

If $\mathcal{F}$ is set to zero, all eigenvalues of $Df(x_0)$ have negative real parts; equivalently, those of $V$ and of $J_4$ have positive real parts.

Result: **holds** (symbolic proof for all positive parameters).

* $V = \left[\begin{matrix}\frac{\alpha c + b \beta}{c} & 0\\- \nu & \frac{b \tau + c \mu}{c}\end{matrix}\right]$: **holds**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}d & 0\\0 & c\end{matrix}\right]$: **holds**
* $V$: $\alpha + \frac{b \beta}{c} > 0$, $\frac{b \tau}{c} + \mu > 0$
* $J_4$: $d > 0$, $c > 0$

### Consequences (Lemma 1)

* $F \ge 0$ (entrywise): **holds**
* $V$ has the Z sign pattern ($V_{ij} \le 0$, $i \ne j$): **holds**
* $\partial \mathcal{V}_i / \partial x_j (x_0) = 0$ for $i \le m < j$: **holds**

If $V$ has the Z pattern and its eigenvalues have positive real parts, $V$ is a non-singular M-matrix and $V^{-1} \ge 0$.

### Conclusion

(A1) to (A5) hold. By Theorem 2, the DFE $x_0$ is locally asymptotically stable if $\mathcal{R}_0 < 1$ and unstable if $\mathcal{R}_0 > 1$.

There is also a text summary and access to each result:

In [4]:
print(report_th)
print()
print("Does Theorem 2 apply?", report_th.holds)
print("Status of each assumption:", report_th.status)
print("(A5):", report_th["A5"].status, "|", report_th["A5"].basis)

van den Driessche & Watmough (2002) assumptions:
  (A1) holds     [symbolic proof for all positive parameters]  non-negative flows
  (A2) holds     [symbolic proof for all positive parameters]  no outflow from an empty compartment
  (A3) holds     [by construction]  no new infections in uninfected compartments
  (A4) holds     [symbolic proof for all positive parameters]  the disease-free subspace is invariant
  (A5) holds     [symbolic proof for all positive parameters]  the DFE is stable without new infections
Theorem 2 applies (R0 is a threshold).

Does Theorem 2 apply? True
Status of each assumption: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}
(A5): holds | symbolic


### The LaTeX report for a paper

`to_latex()` returns the report in LaTeX (requires `amsmath`); with `standalone=True` it is a file that compiles as is; here it is saved in `outputs/05_vdw_assumptions.tex`. It can be written in Spanish with `language="es"`.

In [5]:
tex = report_th.to_latex(standalone=True)
import os
os.makedirs("outputs", exist_ok=True)
with open("outputs/05_vdw_assumptions.tex", "w") as f:
    f.write(tex)
print(tex[:1400], "...")

\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage{amsmath,amssymb}
\usepackage[margin=2cm]{geometry}
\begin{document}

\section*{Assumptions of van den Driessche and Watmough (2002)}

Infected compartments ($i \le m$): $I, V$. No parameter values given: each assumption is proved for every positive parameter value when possible, and otherwise explored with random samples.

\begin{center}
\begin{tabular}{c p{0.34\textwidth} c p{0.3\textwidth}}
\hline
Assumption & Statement & Result & Basis \\
\hline
(A1) & non-negative flows & \textbf{holds} & symbolic proof for all positive parameters \\
(A2) & no outflow from an empty compartment & \textbf{holds} & symbolic proof for all positive parameters \\
(A3) & no new infections in uninfected compartments & \textbf{holds} & by construction \\
(A4) & the disease-free subspace is invariant & \textbf{holds} & symbolic proof for all positive parameters \\
(A5) & the DFE is stable without new infections & \textbf{holds} & symbolic proof 

## 2. Models that violate each assumption

Each model in this section violates **a single** assumption; the other four still hold.

### (A1) Negative new infections

If `new_infections={"I": "beta*S*I*(1 - k*I)"}` is given by hand, that term is negative when $kI > 1$: it is not a rate of new infections. The report shows a counterexample.

In [6]:
text_a1 = '''
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I*(1 - k*I) - (gamma + mu)*I
'''
a1 = R0Model(text_a1, infected=["I"], new_infections={"I": "beta*S*I*(1 - k*I)"})
report_a1 = a1.check_assumptions()
print(report_a1.status)
section(report_a1, "A1")

{'A1': 'fails', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A1) non-negative flows

If $x \ge 0$, then $\mathcal{F}_i,\ \mathcal{V}_i^+,\ \mathcal{V}_i^- \ge 0$ for $i = 1,\dots,n$.

Result: **fails** (random parameter samples).

* $\mathcal{F}_{I} = I S \beta \left(- I k + 1\right) \ge 0$: **fails**, random parameter samples; negative for some $x \ge 0$ in all 40 parameter samples (counterexample: $I = 855.4,\ S = 0.8572,\ \beta = 0.06447,\ k = 0.01121$)
* $\mathcal{V}_i^+$ and $\mathcal{V}_i^-$ collect the negative and positive terms of $\mathcal{V}_i$; they are non-negative when every term has a definite sign.


The automatic decomposition splits $\beta S I - \beta k S I^2$ into $\mathcal{F}_I = \beta S I$ and an outflow $\beta k S I^2$ in $\mathcal{V}_I^-$, and then the five assumptions hold:

In [7]:
print(R0Model(text_a1, infected=["I"]).check_assumptions().status)

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A2) Outflows from an empty compartment

A constant harvest $h$ of susceptibles still acts when $S = 0$, which can make $S$ negative. The library still computes an $\mathcal{R}_0$, but the model does not satisfy (A2).

In [8]:
a2 = R0Model('''
    dS/dt = Lambda - mu*S - h - beta*S*I
    dI/dt = beta*S*I - (gamma + mu)*I
''', infected=["I"])
print("R0 =", a2.R0)
report_a2 = a2.check_assumptions()
print(report_a2.status)
section(report_a2, "A2")

R0 = -beta*(-Lambda + h)/(mu*(gamma + mu))
{'A1': 'holds', 'A2': 'fails', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A2) no outflow from an empty compartment

If $x_i = 0$, then $\mathcal{V}_i^- = 0$.

Result: **fails** (symbolic proof for all positive parameters).

* $\left.\mathcal{V}_{S}^-\right\rvert_{S = 0} = h$: **fails** (counterexample: $h = 0.01121$)
* $\left.\mathcal{V}_{I}^-\right\rvert_{I = 0} = 0$: **holds**


### (A3) By construction

pyR0compute only defines $\mathcal{F}_i$ in the infected compartments, so (A3) always holds. If new infections are assigned to an uninfected compartment, the library rejects them:

In [9]:
try:
    R0Model('''
        dS/dt = Lambda - beta*S*I - mu*S
        dI/dt = beta*S*I - (gamma + mu)*I
    ''', infected=["I"], new_infections=["beta*S*I", "beta*S*I"])
except Exception as error:
    print(type(error).__name__, ":", error)

ModelSpecificationError : A non-zero new-infection term was given for the uninfected compartment 'S'.


### (A4) Relapse from a compartment declared uninfected

If recovered individuals $R$ relapse into $I$ at rate $\varepsilon$ (as in tuberculosis or *P. vivax* malaria), $R$ contains latent infection. Declaring it uninfected violates (A4): in the infection-free subspace ($I = 0$) there is still inflow into $I$ from $R$. Lemma 1 also fails, because $\partial \mathcal{V}_I / \partial R (x_0) = -\varepsilon \neq 0$: the next-generation matrix built only with $I$ ignores the relapse.

In [10]:
text_a4 = '''
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I - (gamma + mu)*I + eps*R
    dR/dt = gamma*I - (mu + eps)*R
'''
a4 = R0Model(text_a4, infected=["I"])
report_a4 = a4.check_assumptions()
print(report_a4.status)
section(report_a4, "A4")

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'fails', 'A5': 'holds'}


### (A4) the disease-free subspace is invariant

If $x \in X_s$, then $\mathcal{F}_i(x) = 0$ and $\mathcal{V}_i^+(x) = 0$ for $i = 1,\dots,m$.

Result: **fails** (symbolic proof for all positive parameters).

* $\left.\mathcal{F}_{I}\right\rvert_{X_s} = 0$: **holds**
* $\left.\mathcal{V}_{I}^+\right\rvert_{X_s} = R eps$: **fails** (counterexample: $R = 10.84,\ eps = 0.01327$)
* $X_s = \{x \ge 0 : x_i = 0,\ i \le m\}$; the uninfected compartments are free, not fixed at $x_0$.


Declaring $R$ infected makes the five assumptions hold, and $\mathcal{R}_0$ changes:
$$
\mathcal{R}_0^{(R\ \text{uninfected})} = \frac{\Lambda\beta}{\mu(\gamma+\mu)}
\qquad\text{versus}\qquad
\mathcal{R}_0^{(R\ \text{infected})} = \frac{\Lambda\beta\,(\varepsilon+\mu)}{\mu^2(\varepsilon+\gamma+\mu)}.
$$
With the values below, ignoring the relapse underestimates $\mathcal{R}_0$ almost threefold.

In [11]:
a4_fixed = R0Model(text_a4, infected=["I", "R"])
print("Assumptions with I and R infected:", a4_fixed.check_assumptions().status)
print("R0 =", sp.factor(a4_fixed.R0))

values_a4 = dict(Lambda=10, beta=0.02, mu=0.1, gamma=0.5, eps=0.3)
print(f"\nR0 with R uninfected: {a4.R0_numeric(values_a4):.4f}")
print(f"R0 with R infected:   {a4_fixed.R0_numeric(values_a4):.4f}")

Assumptions with I and R infected: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


R0 = Lambda*beta*(eps + mu)/(mu**2*(eps + gamma + mu))

R0 with R uninfected: 3.3333
R0 with R infected:   8.8889


### (A5) A DFE that is unstable without infection

With a strong Allee effect, $\dot S = rS(S/K - 1)$, the equilibrium $S = K$ is unstable even without infection. By default the library chooses the DFE with the most nonzero compartments ($S = K$), and the report shows that it fails (A5) and that the other DFE found ($S = 0$) satisfies it.

In [12]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # warning: two DFEs were found
    allee = R0Model('''
        dS/dt = r*S*(S/K - 1) - beta*S*I
        dI/dt = beta*S*I - g*I
    ''', infected=["I"])
report_allee = allee.check_assumptions()
print(report_allee.status)
section(report_allee, "A5")

{'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'fails'}


### (A5) the DFE is stable without new infections

If $\mathcal{F}$ is set to zero, all eigenvalues of $Df(x_0)$ have negative real parts; equivalently, those of $V$ and of $J_4$ have positive real parts.

Result: **fails** (symbolic proof for all positive parameters).

* $V = \left[\begin{matrix}g\end{matrix}\right]$: **holds**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}- r\end{matrix}\right]$: **fails**
* $V$: $g > 0$
* $J_4$: $- r \le 0$

Other disease-free equilibria found (`dfe_candidates`); one can be chosen with `dfe={...}`:

* DFE $S = K$ (the one used): $J_4$ **fails**
* DFE $S = 0$: $J_4$ **holds**


### (A5) $V$ is not an M-matrix: the decomposition matters

If infected cells proliferate ($aI$) and that term is counted as a transition, $V = g - a$. When $a > g$, $V$ has a negative eigenvalue and the formula gives a **negative** $\mathcal{R}_0$, which is meaningless. Without parameter values the assumption is **undecided**, because it depends on $a$ versus $g$; with `values=` it is decided.

In [13]:
text_p = '''
    dS/dt = L - mu*S - beta*S*I
    dI/dt = beta*S*I + a*I - g*I
'''
prolif = R0Model(text_p, infected=["I"])
print("R0 =", prolif.R0)
print("Without values:", prolif.check_assumptions().status["A5"])

values_p = dict(L=1, mu=0.1, beta=0.5, a=0.5, g=0.2)        # a > g
report_p = prolif.check_assumptions(values=values_p)
print("With a = 0.5 > g = 0.2:", report_p.status["A5"])
print("With a = 0.1 < g = 0.2:", prolif.check_assumptions(values=dict(values_p, a=0.1)).status["A5"])
section(report_p, "A5")

R0 = -L*beta/(mu*(a - g))
Without values: undecided
With a = 0.5 > g = 0.2: fails


With a = 0.1 < g = 0.2: holds


### (A5) the DFE is stable without new infections

If $\mathcal{F}$ is set to zero, all eigenvalues of $Df(x_0)$ have negative real parts; equivalently, those of $V$ and of $J_4$ have positive real parts.

Result: **fails** (at the given parameter values).

* $V = \left[\begin{matrix}- a + g\end{matrix}\right]$: **fails**, at the given parameter values
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}\mu\end{matrix}\right]$: **holds**
* $V$: $- a + g$
* eigenvalues of $V$ at the given values: $-0.3$
* $J_4$: $\mu > 0$


If proliferation is counted as a new infection, `new_infections={"I": "beta*S*I + a*I"}`, then $V = g$ and the five assumptions hold for every parameter. We compare both decompositions with the sign of the largest real part of the eigenvalues of the full Jacobian at the DFE:

In [14]:
prolif_F = R0Model(text_p, infected=["I"], new_infections={"I": "beta*S*I + a*I"})
print("Assumptions:", prolif_F.check_assumptions().status)
print("R0 =", sp.simplify(prolif_F.R0), "\n")

f = sp.Matrix([prolif.equations[x] for x in prolif.variables])
J = f.jacobian(list(prolif.variables))
for v in [values_p, dict(values_p, a=0.1), dict(values_p, a=0.1, L=0.01)]:
    sub = {s: v[s.name] for s in prolif.R0.free_symbols}
    r0_v = float(prolif.R0.subs(sub))
    r0_f = float(prolif_F.R0.subs({s: v[s.name] for s in prolif_F.R0.free_symbols}))
    Jn = J.subs(prolif.dfe)
    Jn = Jn.subs({s: v[s.name] for s in Jn.free_symbols})
    abscissa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"a = {v['a']:<4} L = {v['L']:<5} R0 (aI in V) = {r0_v:8.3f}   R0 (aI in F) = {r0_f:7.3f}   max Re(λ) = {abscissa:+.3f}")

Assumptions: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


R0 = (L*beta + a*mu)/(g*mu) 

a = 0.5  L = 1     R0 (aI in V) =  -16.667   R0 (aI in F) =  27.500   max Re(λ) = +5.300
a = 0.1  L = 1     R0 (aI in V) =   50.000   R0 (aI in F) =  25.500   max Re(λ) = +4.900
a = 0.1  L = 0.01  R0 (aI in V) =    0.500   R0 (aI in F) =   0.750   max Re(λ) = -0.050


When the assumptions hold (here, with $aI$ in $\mathcal{F}$, or with $a < g$ in both cases), $\mathcal{R}_0 > 1$ coincides with an unstable DFE and $\mathcal{R}_0 < 1$ with a stable one. The values of $\mathcal{R}_0$ differ between decompositions, but the threshold is the same. With $a > g$ and $aI$ in $\mathcal{V}$, the formula gives $\mathcal{R}_0 < 0$ although the DFE is unstable.

`R0_numeric` returns the spectral radius, which is an absolute value, so it hides the sign. That is why it is worth checking `check_assumptions()`.

## 3. Large models: deciding with `values=`

In the 7-equation model of notebook 03, (A1) to (A4) are proved symbolically. For (A5), the block $\{H, R\}$ of $J_4$ depends on $H^*$ and $R^*$, which are roots of a quadratic, and stability cannot be proved for every parameter. With random samples the DFE is stable in all of them, but that is not a proof, so (A5) stays **undecided**. With parameter values it is decided.

In [15]:
immune = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])

without_values = immune.check_assumptions()
print("Without values:", without_values.status)
print("  (A5):", without_values["A5"].items[1].decision.note["en"])

values = dict(
    g_E=1.0, d_E=0.1, tau_E=2.0, z_VE=1.0, d_I=0.5, tau_I=0.5, z_CI=1.0,
    nu=10.0, d_V=1.0, tau_V=0.5, z_BV=1.0,
    g_H=1.0, d_H=1.0, tau_H=0.5, rho_H=0.3, z_IH=1.0, z_RH=1.0,
    g_C=1.0, d_C=1.0, tau_C=0.5, rho_C=0.3, z_HC=1.0, z_RC=1.0,
    g_B=1.0, d_B=1.0, tau_B=0.5, rho_B=0.3, z_HB=1.0, z_RB=1.0,
    g_R=1.0, d_R=1.0, tau_R=0.5, rho_R=0.3, z_HR=1.0,
)
with_values = immune.check_assumptions(values=values)
print("\nWith values:", with_values.status)
section(with_values, "A5")

Without values: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'undecided'}
  (A5): not proved; stable in all 51 feasible samples (pass values=... to decide at a point)



With values: {'A1': 'holds', 'A2': 'holds', 'A3': 'holds', 'A4': 'holds', 'A5': 'holds'}


### (A5) the DFE is stable without new infections

If $\mathcal{F}$ is set to zero, all eigenvalues of $Df(x_0)$ have negative real parts; equivalently, those of $V$ and of $J_4$ have positive real parts.

Result: **holds** (at the given parameter values).

* $V = \left[\begin{matrix}\frac{C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}}{C^{*} + z_{CI}} & 0\\- \nu & \frac{B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}}{B^{*} + z_{BV}}\end{matrix}\right]$: **holds**
* $J_4 = -\left.\frac{\partial f_u}{\partial x_u}\right\rvert_{x_0} = \left[\begin{matrix}d_{E} & 0 & 0 & 0 & 0\\0 & \frac{R^{*} \rho_{H}}{R^{*} + z_{RH}} + d_{H} & 0 & 0 & H^{*} \left(- \frac{R^{*} \rho_{H}}{\left(R^{*} + z_{RH}\right)^{2}} + \frac{\rho_{H}}{R^{*} + z_{RH}}\right)\\0 & C^{*} \left(\frac{H^{*} \tau_{C}}{\left(H^{*} + z_{HC}\right)^{2}} - \frac{\tau_{C}}{H^{*} + z_{HC}}\right) & - \frac{H^{*} \tau_{C}}{H^{*} + z_{HC}} + \frac{R^{*} \rho_{C}}{R^{*} + z_{RC}} + d_{C} & 0 & C^{*} \left(- \frac{R^{*} \rho_{C}}{\left(R^{*} + z_{RC}\right)^{2}} + \frac{\rho_{C}}{R^{*} + z_{RC}}\right)\\0 & B^{*} \left(\frac{H^{*} \tau_{B}}{\left(H^{*} + z_{HB}\right)^{2}} - \frac{\tau_{B}}{H^{*} + z_{HB}}\right) & 0 & - \frac{H^{*} \tau_{B}}{H^{*} + z_{HB}} + \frac{R^{*} \rho_{B}}{R^{*} + z_{RB}} + d_{B} & B^{*} \left(- \frac{R^{*} \rho_{B}}{\left(R^{*} + z_{RB}\right)^{2}} + \frac{\rho_{B}}{R^{*} + z_{RB}}\right)\\0 & R^{*} \left(\frac{H^{*} \tau_{R}}{\left(H^{*} + z_{HR}\right)^{2}} - \frac{\tau_{R}}{H^{*} + z_{HR}}\right) & 0 & 0 & - \frac{H^{*} \tau_{R}}{H^{*} + z_{HR}} + d_{R} + \rho_{R}\end{matrix}\right]$: **holds**, at the given parameter values
* $V$: $\frac{C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}}{C^{*} + z_{CI}} > 0$, $\frac{B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}}{B^{*} + z_{BV}} > 0$
* $J_4$: $d_{E} > 0$, $\operatorname{tr} = - \frac{H^{*} \tau_{R}}{H^{*} + z_{HR}} + \frac{R^{*} \rho_{H}}{R^{*} + z_{RH}} + d_{H} + d_{R} + \rho_{R},\ \det = \frac{\left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} d_{R} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} \rho_{R} - \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{H} \tau_{R} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} d_{R} \rho_{H} + \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} \rho_{H} \rho_{R} - \left(H^{*}\right)^{2} \left(R^{*}\right)^{2} \rho_{H} \tau_{R} + 2 \left(H^{*}\right)^{2} R^{*} d_{H} d_{R} z_{RH} + 2 \left(H^{*}\right)^{2} R^{*} d_{H} \rho_{R} z_{RH} - 2 \left(H^{*}\right)^{2} R^{*} d_{H} \tau_{R} z_{RH} + \left(H^{*}\right)^{2} R^{*} d_{R} \rho_{H} z_{RH} + \left(H^{*}\right)^{2} R^{*} \rho_{H} \rho_{R} z_{RH} - \left(H^{*}\right)^{2} R^{*} \rho_{H} \tau_{R} z_{RH} + \left(H^{*}\right)^{2} d_{H} d_{R} z_{RH}^{2} + \left(H^{*}\right)^{2} d_{H} \rho_{R} z_{RH}^{2} - \left(H^{*}\right)^{2} d_{H} \tau_{R} z_{RH}^{2} + 2 H^{*} \left(R^{*}\right)^{2} d_{H} d_{R} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} d_{H} \rho_{R} z_{HR} - H^{*} \left(R^{*}\right)^{2} d_{H} \tau_{R} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} d_{R} \rho_{H} z_{HR} + 2 H^{*} \left(R^{*}\right)^{2} \rho_{H} \rho_{R} z_{HR} - H^{*} \left(R^{*}\right)^{2} \rho_{H} \tau_{R} z_{HR} + 4 H^{*} R^{*} d_{H} d_{R} z_{HR} z_{RH} + 4 H^{*} R^{*} d_{H} \rho_{R} z_{HR} z_{RH} - 2 H^{*} R^{*} d_{H} \tau_{R} z_{HR} z_{RH} + 2 H^{*} R^{*} d_{R} \rho_{H} z_{HR} z_{RH} + 2 H^{*} R^{*} \rho_{H} \rho_{R} z_{HR} z_{RH} + 2 H^{*} d_{H} d_{R} z_{HR} z_{RH}^{2} + 2 H^{*} d_{H} \rho_{R} z_{HR} z_{RH}^{2} - H^{*} d_{H} \tau_{R} z_{HR} z_{RH}^{2} + \left(R^{*}\right)^{2} d_{H} d_{R} z_{HR}^{2} + \left(R^{*}\right)^{2} d_{H} \rho_{R} z_{HR}^{2} + \left(R^{*}\right)^{2} d_{R} \rho_{H} z_{HR}^{2} + \left(R^{*}\right)^{2} \rho_{H} \rho_{R} z_{HR}^{2} + 2 R^{*} d_{H} d_{R} z_{HR}^{2} z_{RH} + 2 R^{*} d_{H} \rho_{R} z_{HR}^{2} z_{RH} + R^{*} d_{R} \rho_{H} z_{HR}^{2} z_{RH} + R^{*} \rho_{H} \rho_{R} z_{HR}^{2} z_{RH} + d_{H} d_{R} z_{HR}^{2} z_{RH}^{2} + d_{H} \rho_{R} z_{HR}^{2} z_{RH}^{2}}{\left(H^{*}\right)^{2} \left(R^{*}\right)^{2} + 2 \left(H^{*}\right)^{2} R^{*} z_{RH} + \left(H^{*}\right)^{2} z_{RH}^{2} + 2 H^{*} \left(R^{*}\right)^{2} z_{HR} + 4 H^{*} R^{*} z_{HR} z_{RH} + 2 H^{*} z_{HR} z_{RH}^{2} + \left(R^{*}\right)^{2} z_{HR}^{2} + 2 R^{*} z_{HR}^{2} z_{RH} + z_{HR}^{2} z_{RH}^{2}}$, $- \frac{H^{*} \tau_{C}}{H^{*} + z_{HC}} + \frac{R^{*} \rho_{C}}{R^{*} + z_{RC}} + d_{C}$, $- \frac{H^{*} \tau_{B}}{H^{*} + z_{HB}} + \frac{R^{*} \rho_{B}}{R^{*} + z_{RB}} + d_{B}$
* eigenvalues of $J_4$ at the given values: $0.9121, 0.9121, 1.106 + 0.08827i, 1.106 - 0.08827i, 0.1$


## Summary

| Model | Assumption violated | What the report shows |
|---|:--:|---|
| $\mathcal{F}_I = \beta S I (1 - kI)$ given by hand | (A1) | counterexample with $kI > 1$ |
| Constant harvest $h$ | (A2) | $\mathcal{V}_S^- = h \ne 0$ when $S = 0$ |
| Assigning $\mathcal{F}$ to an uninfected compartment | (A3) | the library rejects it |
| Relapse $\varepsilon R \to I$ with $R$ uninfected | (A4) | $\mathcal{V}_I^+ = \varepsilon R$ in $X_s$; declare $R$ infected |
| Allee effect | (A5) | $J_4 = -r$; the DFE $S = 0$ satisfies it |
| Proliferation $aI$ in $\mathcal{V}$ with $a > g$ | (A5) | $V = g - a < 0$; put $aI$ in $\mathcal{F}$ |

If `report.holds` is `True`, the computed $\mathcal{R}_0$ is a threshold by Theorem 2 of van den Driessche and Watmough (2002).